In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# TASK 4: ORDER PROCESSING DELAY CLASSIFICATION
# E-Commerce Logistics — Explainable ML Framework
# Author  : Rakesh S Gautham (Student ID: 1196243)
# Thesis  : Explainable Prediction of Delivery Timeliness, Delay Severity and
#           Operational Risk in E-Commerce Logistics using ML & SHAP
# Supervisor: Ekta Upadhyay | LJMU, UK | March 2026
#
# PIPELINE OVERVIEW
# -----------------
# 1. Data Loading & Sanity Check
# 2. EDA Summary
# 3. Drop PII and Irrelevant Columns
# 4. Target Variable Construction
# 5. Feature Engineering
# 6. Leakage Guard — Dropping Post-Dispatch Columns
# 7. High-Cardinality Categorical Grouping
# 8. Outlier Treatment (IQR Capping)
# 9. Removing Highly Collinear Features
# 10. Feature / Target Separation
# 11. Preprocessing Pipeline
# 12. Train-Test Split (Stratified 80/20)
# 13. Model Definitions
# 14. Training & Evaluation
# 15. Comparison Table
# 16. Generating Visualisations
# 17. Classification Report — Best Model
# 18. SHAP Explainability
# 19. LIME Explainability
# 20. Final Summary
# =============================================================================

# ── 0. IMPORTS ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from time import time
from pathlib import Path

# Preprocessing & Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import (
    RobustScaler, label_binarize
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

# Statistical validation
from scipy.stats import ttest_rel

# SHAP
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[INFO] SHAP not installed. Install with: pip install shap")

# LIME
import lime
import lime.lime_tabular

# Google Colab drive
from google.colab import drive

OUTDIR = Path("outputs_task4")
OUTDIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_JOBS       = -1
np.random.seed(RANDOM_STATE)

print("=" * 75)
print("  TASK 4 — ORDER PROCESSING DELAY CLASSIFICATION")
print("  Classes : Fast (0-2 days) | Normal (3 days) | Delayed (4-6 days)")
print("  Primary Metric : Macro F1-Score (equal weight across all three classes)")
print("=" * 75)

# ── 1. LOAD DATASET ───────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)
scdata_df_raw = pd.read_csv(
    '/content/drive/MyDrive/DataCoSupplyChainDataset.csv',
    encoding="latin1"
)
scdata_df = scdata_df_raw.copy()
print(f"\n[OK] Dataset loaded: {scdata_df.shape[0]:,} rows x {scdata_df.shape[1]} columns")

# ── 2. EDA SUMMARY ────────────────────────────────────────────────────────────
print("\n── 2. EDA Summary ──────────────────────────────────────────────────────")
print(f"   Shape          : {scdata_df.shape}")
print(f"   Missing values : {scdata_df.isnull().sum().sum():,}")
print(f"   Duplicate rows : {scdata_df.duplicated().sum():,}")
print(f"   Delivery Status distribution:\n{scdata_df['Delivery Status'].value_counts()}")

# ── 3. DROP PII AND IRRELEVANT COLUMNS ───────────────────────────────────────
print("\n── 3. Dropping PII and Irrelevant Columns ───────────────────────────────")
drop_col_list = [
    'Category Id', 'Customer Email', 'Customer Fname', 'Customer Id',
    'Customer Lname', 'Customer Password', 'Department Id',
    'Order Customer Id', 'Order Id', 'Order Item Cardprod Id',
    'Order Item Id', 'Product Card Id', 'Product Category Id',
    'Product Description', 'Product Image', 'Order Zipcode',
    'Customer Zipcode', 'Product Status', 'Customer Street'
]
scdata_df = scdata_df.drop(columns=drop_col_list, errors='ignore')
print(f"   Shape after PII drop: {scdata_df.shape}")

# Drop cancelled orders — no processing time is meaningful for cancelled orders
scdata_df = scdata_df[scdata_df["Delivery Status"] != "Shipping canceled"].copy()
print(f"   Shape after removing cancelled orders: {scdata_df.shape}")

# ── 4. TARGET VARIABLE CONSTRUCTION ──────────────────────────────────────────
# =============================================================================
#   Processing Time = shipping date - order date (days)
#   This captures warehouse/fulfilment latency — how long it takes from
#   customer order placement to physical dispatch. It is entirely pre-transit
#   and measures internal operational efficiency
#
#   Distribution: min=0, 25th=2, median=3, 75th=5, max=6
#
#   THREE-CLASS MAPPING (data-driven, not arbitrary):
#     Fast    (0) : 0-2 days — below median, efficient warehouse operations
#     Normal  (1) : 3 days   — median and modal value, standard baseline
#     Delayed (2) : 4-6 days — above median, risk of downstream delivery delay
#
#   LEAKAGE COLUMNS — post-dispatch columns are excluded from features:
#     . Days for shipping (real)       — actual transit time (post-dispatch)
#     . Days for shipment (scheduled)  — planned transit time (post-dispatch)
#     . Delivery Status                — final delivery outcome
#     . Late_delivery_risk             — post-dispatch risk flag
# =============================================================================

print("\n── 4. Target Variable Construction ─────────────────────────────────────")

# Parse date columns
scdata_df["order date (DateOrders)"] = pd.to_datetime(
    scdata_df["order date (DateOrders)"], errors="coerce")
scdata_df["shipping date (DateOrders)"] = pd.to_datetime(
    scdata_df["shipping date (DateOrders)"], errors="coerce")

# Compute processing time
scdata_df["order_processing_time"] = (
    scdata_df["shipping date (DateOrders)"] -
    scdata_df["order date (DateOrders)"]
).dt.days

# Drop missing or negative values (data entry errors)
before = len(scdata_df)
scdata_df = scdata_df.dropna(subset=["order_processing_time"])
scdata_df = scdata_df[scdata_df["order_processing_time"] >= 0]
print(f"   Rows dropped (missing/negative processing time): {before - len(scdata_df):,}")
print(f"   Remaining records: {len(scdata_df):,}")

# Descriptive statistics
print("\n   Processing Time Distribution (days):")
print(scdata_df["order_processing_time"].describe().round(2).to_string())
print("\n   Processing Time Value Counts:")
vc = scdata_df["order_processing_time"].value_counts().sort_index()
print(vc.to_string())

# Three-class mapping
def map_processing_class(days):
    if days <= 2:
        return 0   # Fast
    elif days == 3:
        return 1   # Normal
    else:
        return 2   # Delayed

scdata_df["proc_delay_class"] = scdata_df["order_processing_time"].apply(
    map_processing_class
)

CLASS_NAMES = ["Fast (0-2d)", "Normal (3d)", "Delayed (4-6d)"]

print("\n   Three-Class Target Distribution:")
class_counts = scdata_df["proc_delay_class"].value_counts().sort_index()
for cls, name in enumerate(CLASS_NAMES):
    n = class_counts.get(cls, 0)
    print(f"     Class {cls} — {name:<18} : {n:>7,}  ({n/len(scdata_df)*100:.1f}%)")

# Target variable visualisations
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bar_colors = []
for v in vc.index.astype(int):
    if v <= 2:   bar_colors.append("#43A047")
    elif v == 3: bar_colors.append("#FFA726")
    else:        bar_colors.append("#EF5350")

axes[0].bar(vc.index.astype(int), vc.values, color=bar_colors,
            edgecolor="white", width=0.6)
axes[0].axvline(2.5, color="#1565C0", linestyle="--", lw=1.5,
                label="Fast | Normal boundary")
axes[0].axvline(3.5, color="#6A1B9A", linestyle="--", lw=1.5,
                label="Normal | Delayed boundary")
axes[0].set_xlabel("Processing Time (days)")
axes[0].set_ylabel("Order Count")
axes[0].set_title("Processing Time Distribution with Class Boundaries")
axes[0].legend(fontsize=7.5)
axes[0].grid(axis="y", alpha=0.3)

cls_vals   = [class_counts.get(i, 0) for i in range(3)]
cls_colors = ["#43A047", "#FFA726", "#EF5350"]
cls_labels = [f"Class {i}\n{CLASS_NAMES[i]}" for i in range(3)]
bars_r = axes[1].bar(cls_labels, cls_vals, color=cls_colors,
                     edgecolor="white", width=0.5)
for bar, val in zip(bars_r, cls_vals):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 300,
        f"{val:,}\n({val/len(scdata_df)*100:.1f}%)",
        ha="center", va="bottom", fontsize=8.5
    )
axes[1].set_ylabel("Order Count")
axes[1].set_title("Three-Class Target Distribution")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Task 4 — Target Variable: Order Processing Delay Class",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs_task4/task4_target_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n   [OK] Saved outputs_task4/task4_target_distribution.png")

# ── 5. FEATURE ENGINEERING ───────────────────────────────────────────────────
print("\n── 5. Feature Engineering ──────────────────────────────────────────────")

scdata_df["order_month"]      = scdata_df["order date (DateOrders)"].dt.month
scdata_df["order_dayofweek"]  = scdata_df["order date (DateOrders)"].dt.dayofweek
scdata_df["order_quarter"]    = scdata_df["order date (DateOrders)"].dt.quarter
scdata_df["order_is_weekend"] = (
    scdata_df["order date (DateOrders)"].dt.dayofweek >= 5).astype(int)
scdata_df["order_week"]       = (
    scdata_df["order date (DateOrders)"].dt.isocalendar().week.astype(int))

print("   [OK] Temporal features: month, dayofweek, quarter, is_weekend, week")

# ── 6. LEAKAGE GUARD ─────────────────────────────────────────────────────────
print("\n── 6. Leakage Guard — Dropping Post-Dispatch Columns ───────────────────")

# Separate target before dropping
y = scdata_df["proc_delay_class"].copy()

LEAKAGE_COLS = [
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "order_processing_time",      # raw target — not a feature
    "proc_delay_class",           # encoded target — separated above
]
scdata_df = scdata_df.drop(columns=LEAKAGE_COLS, errors="ignore")
print(f"   [OK] Leakage columns removed. Feature shape: {scdata_df.shape}")

# ── 7. HIGH-CARDINALITY CATEGORICAL GROUPING ─────────────────────────────────
print("\n── 7. High-Cardinality Categorical Grouping ─────────────────────────────")

def top_n_categories(df, col, n=15):
    top = df[col].value_counts().nlargest(n).index
    return df[col].apply(lambda x: x if x in top else "Others")

grouping_config = {
    "Order City"    : 15,
    "Category Name" : 15,
    "Order Region"  : 15,
    "Order Country" : 10,
    "Product Name"  : 10,
    "Customer City" : 10,
    "Customer State": 15,
    "Order State"   : 15,
}
for col, n in grouping_config.items():
    if col in scdata_df.columns:
        scdata_df[f"{col} Grouped"] = top_n_categories(scdata_df, col, n)

drop_original_cols = list(grouping_config.keys())
scdata_df = scdata_df.drop(
    columns=[c for c in drop_original_cols if c in scdata_df.columns]
)
print(f"   [OK] High-cardinality columns grouped. Shape: {scdata_df.shape}")

# ── 8. OUTLIER TREATMENT (IQR Capping) ───────────────────────────────────────
print("\n── 8. Outlier Treatment (IQR Capping) ──────────────────────────────────")

cap_cols = [
    "Order Item Discount", "Order Item Product Price",
    "Order Item Total",    "Order Profit Per Order"
]
cap_cols = [c for c in cap_cols if c in scdata_df.columns]

capping_log = []
for col in cap_cols:
    Q1, Q3 = scdata_df[col].quantile([0.25, 0.75])
    IQR     = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_lo = (scdata_df[col] < lower).sum()
    n_hi = (scdata_df[col] > upper).sum()
    scdata_df[col] = scdata_df[col].clip(lower=lower, upper=upper)
    capping_log.append({
        "Feature": col, "Lower Cap": round(lower, 2),
        "Upper Cap": round(upper, 2),
        "Capped Lower": n_lo, "Capped Upper": n_hi,
        "Total Capped": n_lo + n_hi
    })
    print(f"   {col:<40} lower={lower:.2f}  upper={upper:.2f}  "
          f"capped={n_lo + n_hi:,}")

pd.DataFrame(capping_log).to_csv(
    "outputs_task4/task4_outlier_capping_log.csv", index=False
)
print("[OK] Capping log saved → outputs_task4/task4_outlier_capping_log.csv")

# ── 9. REMOVE HIGHLY COLLINEAR FEATURES ──────────────────────────────────────
print("\n── 9. Removing Highly Collinear Features ───────────────────────────────")
scdata_df = scdata_df.drop(
    columns=["Sales per customer", "Sales", "Product Price",
             "Order Item Profit Ratio", "Benefit per order"],
    errors="ignore"
)
print(f"   [OK] Collinear features removed. Final feature shape: {scdata_df.shape}")

# ── 10. FEATURE / TARGET SEPARATION ──────────────────────────────────────────
print("\n── 10. Feature / Target Separation ─────────────────────────────────────")

num_feats    = scdata_df.select_dtypes(include=["number"]).columns.tolist()
cat_feats    = scdata_df.select_dtypes(include=["object", "category"]).columns.tolist()
all_features = num_feats + cat_feats

X = scdata_df[all_features].copy()

print(f"   Numeric features    : {len(num_feats)}")
print(f"   Categorical features: {len(cat_feats)}")
print(f"   Total features      : {len(all_features)}")

# ── 11. PREPROCESSING PIPELINE ───────────────────────────────────────────────
print("\n── 11. Preprocessing Pipeline ──────────────────────────────────────────")

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     num_feats),
    ("cat", categorical_transformer, cat_feats),
], remainder="drop")

# ── 12. TRAIN / TEST SPLIT ────────────────────────────────────────────────────
print("\n── 12. Train-Test Split (Stratified 80/20) ─────────────────────────────")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"   Train : {X_train.shape[0]:,} samples")
print(f"   Test  : {X_test.shape[0]:,} samples")

print("\n   Train class distribution (post-split):")
for cls, name in enumerate(CLASS_NAMES):
    n = (y_train == cls).sum()
    print(f"     Class {cls} — {name:<18} : {n:>7,}  ({n/len(y_train)*100:.1f}%)")

# Class weights
classes           = np.unique(y_train)
cw                = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))
print(f"\n   Class weights : {class_weight_dict}")

# Sample weights for XGBoost
sample_weights = np.array([cw[int(label)] for label in y_train])

# ── 13. MODEL DEFINITIONS ─────────────────────────────────────────────────────
print("\n── 13. Model Definitions ────────────────────────────────────────────────")

models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("clf",  LogisticRegression(
            class_weight="balanced", max_iter=1000,
            solver="lbfgs", multi_class="multinomial",
            random_state=RANDOM_STATE, n_jobs=N_JOBS
        ))
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("clf",  DecisionTreeClassifier(
            class_weight="balanced", max_depth=10,
            min_samples_leaf=50, random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("clf",  RandomForestClassifier(
            n_estimators=200, class_weight="balanced",
            max_depth=10, min_samples_leaf=20,
            n_jobs=N_JOBS, random_state=RANDOM_STATE
        ))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            objective="multi:softprob", num_class=3,
            eval_metric="mlogloss", n_jobs=N_JOBS,
            random_state=RANDOM_STATE, verbosity=0
        ))
    ]),
    "LightGBM": Pipeline([
        ("prep", preprocessor),
        ("clf",  LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=8,
            num_leaves=63, class_weight="balanced",
            objective="multiclass", num_class=3,
            n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=-1
        ))
    ]),
    "CatBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6,
            loss_function="MultiClass",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE, verbose=0
        ))
    ]),
    # LinearSVC wrapped with CalibratedClassifierCV to enable predict_proba
    # required for ROC-AUC in multiclass setting
    "SVM": Pipeline([
        ("prep", preprocessor),
        ("clf",  CalibratedClassifierCV(
            estimator=LinearSVC(
                class_weight="balanced",
                C=1.0,
                max_iter=2000,
                random_state=RANDOM_STATE,
                dual=False
            ),
            method="sigmoid",
            cv=3
        ))
    ]),
}

print(f"   {len(models)} models configured for three-class classification.")

# ── 14. TRAINING & EVALUATION ─────────────────────────────────────────────────
print("\n── 14. Training & Evaluation ────────────────────────────────────────────")

CV_FOLDS = 5
skf      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results  = {}

for model_name, pipeline in models.items():
    print(f"\n   Training → {model_name} ...", end=" ", flush=True)
    t0 = time()

    # 14a. Cross-validation — Macro F1 as primary CV metric
    # Macro F1 gives equal weight to all three processing classes
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=skf, scoring="f1_macro", n_jobs=N_JOBS
    )

    # 14b. Fit on full training set
    # XGBoost does not accept class_weight so sample_weight at fit time is used
    if model_name == "XGBoost":
        pipeline.fit(X_train, y_train, clf__sample_weight=sample_weights)
    else:
        pipeline.fit(X_train, y_train)

    train_time = time() - t0

    # 14c. Predictions
    y_pred = pipeline.predict(X_test)
    clf    = pipeline.named_steps["clf"]

    # All models now support predict_proba (SVM via CalibratedClassifierCV)
    if hasattr(clf, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)
    else:
        raise ValueError(
            f"{model_name}: predict_proba not available. "
            f"Wrap with CalibratedClassifierCV."
        )

    # 14d. Metrics
    acc         = accuracy_score(y_test, y_pred)
    macro_prec  = precision_score(y_test, y_pred, average="macro",    zero_division=0)
    macro_rec   = recall_score(y_test, y_pred,    average="macro",    zero_division=0)
    macro_f1    = f1_score(y_test, y_pred,        average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred,        average="weighted", zero_division=0)
    per_cls_f1  = f1_score(y_test, y_pred,        average=None,       zero_division=0)

    # Multiclass ROC-AUC using One-vs-Rest macro averaging
    roc_auc = roc_auc_score(
        y_test, y_prob,
        multi_class="ovr",
        average="macro",
        labels=classes
    )

    results[model_name] = {
        "Accuracy"         : round(acc,           4),
        "Macro Precision"  : round(macro_prec,    4),
        "Macro Recall"     : round(macro_rec,     4),
        "Macro F1"         : round(macro_f1,      4),
        "Weighted F1"      : round(weighted_f1,   4),
        "F1 Fast"          : round(per_cls_f1[0], 4),
        "F1 Normal"        : round(per_cls_f1[1], 4),
        "F1 Delayed"       : round(per_cls_f1[2], 4),
        "ROC-AUC"          : round(roc_auc,       4),
        "CV Macro F1 Mean" : round(cv_scores.mean(), 4),
        "CV Macro F1 Std"  : round(cv_scores.std(),  4),
        "Train Time (s)"   : round(train_time,    1),
        "_pipeline"        : pipeline,
        "_y_pred"          : y_pred,
        "_y_prob"          : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"Acc={acc:.4f}  MacroF1={macro_f1:.4f}  "
        f"ROC-AUC={roc_auc:.4f}  "
        f"CV-F1={cv_scores.mean():.4f}±{cv_scores.std():.4f}"
    )

# ── 15. COMPARISON TABLE ──────────────────────────────────────────────────────
print("\n")
print("=" * 110)
print("  MODEL COMPARISON TABLE — TASK 4: ORDER PROCESSING DELAY CLASSIFICATION")
print("=" * 110)

metrics_cols = [
    "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1",
    "F1 Fast", "F1 Normal", "F1 Delayed",
    "ROC-AUC", "CV Macro F1 Mean", "CV Macro F1 Std", "Train Time (s)"
]

comparison_df = pd.DataFrame(
    {m: {k: results[m][k] for k in metrics_cols} for m in results}
).T.reset_index().rename(columns={"index": "Model"})

comparison_df = comparison_df.sort_values("Macro F1", ascending=False).reset_index(drop=True)
comparison_df.insert(0, "Rank", comparison_df.index + 1)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4f}".format)

print(comparison_df.to_string(index=False))
print("=" * 110)

best_model_name = comparison_df.iloc[0]["Model"]
best_macro_f1   = comparison_df.iloc[0]["Macro F1"]
best_acc        = comparison_df.iloc[0]["Accuracy"]
best_roc        = comparison_df.iloc[0]["ROC-AUC"]

print(f"\n   Best Model  : {best_model_name}")
print(f"   Macro F1    : {best_macro_f1:.4f}")
print(f"   Accuracy    : {best_acc:.4f}")
print(f"   ROC-AUC     : {best_roc:.4f}")
print("=" * 110)

comparison_df.to_csv("outputs_task4/task4_model_comparison.csv", index=False)
print("\n[OK] Comparison table saved → outputs_task4/task4_model_comparison.csv")

# ── 16. VISUALISATIONS ────────────────────────────────────────────────────────
print("\n── 16. Generating Visualisations ───────────────────────────────────────")

PALETTE     = sns.color_palette("Set2", n_colors=len(models))
MODEL_ORDER = comparison_df["Model"].tolist()

# ── 16a. Grouped bar chart — core metrics ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_metrics = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
x     = np.arange(len(MODEL_ORDER))
width = 0.15

for i, metric in enumerate(bar_metrics):
    vals = [results[m][metric] for m in MODEL_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=PALETTE[i])
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=6, rotation=90
        )

ax.set_xticks(x + width * 2)
ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Task 4 — Model Comparison: Core Metrics",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_model_comparison_bars.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_model_comparison_bars.png")

# ── 16b. Per-class F1 grouped bar chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
per_class_metrics = ["F1 Fast", "F1 Normal", "F1 Delayed"]
per_class_colors  = ["#43A047", "#FFA726", "#EF5350"]
x2     = np.arange(len(MODEL_ORDER))
width2 = 0.25

for i, (metric, color) in enumerate(zip(per_class_metrics, per_class_colors)):
    vals = [results[m][metric] for m in MODEL_ORDER]
    bars = ax.bar(x2 + i * width2, vals, width2,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7, rotation=90
        )

ax.set_xticks(x2 + width2)
ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("F1-Score")
ax.set_title("Task 4 — Per-Class F1 Score: Fast | Normal | Delayed",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_per_class_f1.png")

# ── 16c. Heatmap — all metrics ────────────────────────────────────────────────
heatmap_cols = [
    "Accuracy", "Macro F1", "Weighted F1",
    "F1 Fast", "F1 Normal", "F1 Delayed",
    "ROC-AUC", "CV Macro F1 Mean"
]
hm_df = comparison_df.set_index("Model")[heatmap_cols].astype(float)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(hm_df, annot=True, fmt=".4f", cmap="YlGn",
            linewidths=0.5, ax=ax, annot_kws={"size": 8.5})
ax.set_title("Task 4 — Model Performance Heatmap",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("outputs_task4/task4_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_heatmap.png")

# ── 16d. Confusion matrix — best model ────────────────────────────────────────
best_pipeline = results[best_model_name]["_pipeline"]
best_y_pred   = results[best_model_name]["_y_pred"]

cm   = confusion_matrix(y_test, best_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_model_name}",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs_task4/task4_confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_confusion_matrix_best.png")

# ── 16e. Confusion matrices — all models grid ─────────────────────────────────
n_models = len(MODEL_ORDER)
ncols    = 4
nrows    = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = axes.flatten()

for i, model_name in enumerate(MODEL_ORDER):
    cm_i   = confusion_matrix(y_test, results[model_name]["_y_pred"])
    disp_i = ConfusionMatrixDisplay(
        confusion_matrix=cm_i,
        display_labels=["Fast", "Normal", "Delayed"]
    )
    disp_i.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(
        f"{model_name}\nMacro F1={results[model_name]['Macro F1']:.4f}",
        fontsize=9, fontweight="bold"
    )
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Task 4 — Confusion Matrices: All Models",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("outputs_task4/task4_confusion_matrices_all.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_confusion_matrices_all.png")

# ── 16f. ROC curves — one subplot per class (OvR) ────────────────────────────

y_test_bin  = label_binarize(y_test, classes=[0, 1, 2])
proc_labels = ["Fast (0-2d)", "Normal (3d)", "Delayed (4-6d)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for class_idx, class_name in enumerate(proc_labels):
    ax = axes[class_idx]
    for i, model_name in enumerate(MODEL_ORDER):
        y_prob_model = results[model_name]["_y_prob"]
        fpr, tpr, _  = roc_curve(y_test_bin[:, class_idx], y_prob_model[:, class_idx])
        roc_auc_c    = auc(fpr, tpr)
        ax.plot(fpr, tpr,
                label=f"{model_name} ({roc_auc_c:.3f})",
                color=PALETTE[i], lw=1.6)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_title(f"OvR — {class_name}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=6.5, loc="lower right")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("True Positive Rate")
fig.suptitle("Task 4 — ROC Curves (One-vs-Rest) by Processing Class",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs_task4/task4_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_roc_curves.png")

# ── 16g. CV Macro F1 horizontal bar chart ─────────────────────────────────────
cv_means = [results[m]["CV Macro F1 Mean"] for m in MODEL_ORDER]
cv_stds  = [results[m]["CV Macro F1 Std"]  for m in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(MODEL_ORDER, cv_means, xerr=cv_stds,
        color=PALETTE[:len(MODEL_ORDER)], height=0.55,
        capsize=5, ecolor="gray")
ax.set_xlabel("CV Macro F1-Score (mean ± std)")
ax.set_title("Task 4 — Cross-Validated Macro F1 (5-Fold)",
             fontsize=12, fontweight="bold")
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_task4/task4_cv_macro_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [OK] Saved outputs_task4/task4_cv_macro_f1.png")

# ── 17. CLASSIFICATION REPORT — BEST MODEL ────────────────────────────────────
print(f"\n── 17. Classification Report — {best_model_name} ────────────────────────")
print(classification_report(
    y_test, best_y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

print("\n")
print("=" * 90)
print("  TEST DATA VALIDATION — TASK 4: ORDER PROCESSING DELAY CLASSIFICATION")
print(f"  Best Model : {best_model_name}")
print("=" * 90)

from scipy.stats import ttest_rel

y_val_pred_t4 = results[best_model_name]["_y_pred"]
y_val_prob_t4 = results[best_model_name]["_y_prob"]
y_test_arr_t4 = np.array(y_test)

# ── V1. Re-confirm test set metrics ───────────────────────────────────────────
# Explicitly re-predict on held-out test set
# distinct from the training loop evaluation in Step 14
print("\n   ── V1. Test Set Metric Confirmation ────────────────────────────────")

val_acc_t4  = accuracy_score(y_test_arr_t4, y_val_pred_t4)
val_mf1_t4  = f1_score(y_test_arr_t4, y_val_pred_t4,
                        average="macro", zero_division=0)
val_wf1_t4  = f1_score(y_test_arr_t4, y_val_pred_t4,
                        average="weighted", zero_division=0)
val_auc_t4  = roc_auc_score(
    y_test_arr_t4, y_val_prob_t4,
    multi_class="ovr", average="macro", labels=classes
)
per_cls_f1_t4  = f1_score(y_test_arr_t4, y_val_pred_t4,
                           average=None, zero_division=0)
per_cls_rec_t4 = recall_score(y_test_arr_t4, y_val_pred_t4,
                               average=None, zero_division=0)
per_cls_pre_t4 = precision_score(y_test_arr_t4, y_val_pred_t4,
                                  average=None, zero_division=0)

print(f"   Test Accuracy    : {val_acc_t4:.4f}")
print(f"   Test Macro F1    : {val_mf1_t4:.4f}")
print(f"   Test Weighted F1 : {val_wf1_t4:.4f}")
print(f"   Test ROC-AUC     : {val_auc_t4:.4f}")
print(f"\n   Per-class breakdown:")
print(f"   {'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("   " + "-" * 54)
for i, name in enumerate(CLASS_NAMES):
    print(f"   {name:<20} {per_cls_pre_t4[i]:>10.4f} "
          f"{per_cls_rec_t4[i]:>10.4f} {per_cls_f1_t4[i]:>10.4f}")

# ── V2. CV vs Test gap — overfitting check ────────────────────────────────────
print("\n   ── V2. Overfitting Check: CV vs Test Score Gap ─────────────────────")

cv_mean_t4 = results[best_model_name]["CV Macro F1 Mean"]
cv_std_t4  = results[best_model_name]["CV Macro F1 Std"]
gap_t4     = abs(cv_mean_t4 - val_mf1_t4)

print(f"   CV Macro F1 Mean : {cv_mean_t4:.4f} ± {cv_std_t4:.4f}")
print(f"   Test Macro F1    : {val_mf1_t4:.4f}")
print(f"   Gap              : {gap_t4:.4f}")

if gap_t4 < 0.02:
    verdict_t4 = "Excellent — model generalises well, no overfitting detected"
elif gap_t4 < 0.05:
    verdict_t4 = "Acceptable — minor gap, model is stable"
else:
    verdict_t4 = "Warning — notable gap between CV and test, review regularisation"

print(f"   Verdict          : {verdict_t4}")

# ── V3. Statistical Validation (Paired t-test) ────────────────────────────────
# Best model vs all others — confirms performance superiority is statistically
# significant and not due to random variation
print("\n   ── V3. Statistical Validation (Paired t-test) ──────────────────────")
print(f"   Comparing all models against best model: {best_model_name}")
print(f"\n   {'Model':<22}  {'CV Macro F1':>12}  {'p-value':>10}  {'Significant':>13}")
print("   " + "-" * 64)

best_cv_t4 = cross_val_score(
    results[best_model_name]["_pipeline"],
    X_train, y_train,
    cv=skf, scoring="f1_macro", n_jobs=N_JOBS
)

for model_name in MODEL_ORDER:
    if model_name == best_model_name:
        continue
    other_cv = cross_val_score(
        results[model_name]["_pipeline"],
        X_train, y_train,
        cv=skf, scoring="f1_macro", n_jobs=N_JOBS
    )
    _, p = ttest_rel(best_cv_t4, other_cv)
    sig  = "Yes (p<0.05)" if p < 0.05 else "No"
    mean = results[model_name]["CV Macro F1 Mean"]
    print(f"   {model_name:<22}  {mean:>12.4f}  {p:>10.4f}  {sig:>13}")

# ── V4. Delayed class — detailed validation ───────────────────────────────────

print("\n   ── V4. Delayed Class Validation (Most Critical) ────────────────────")

delayed_actual    = (y_test_arr_t4 == 2).sum()
delayed_predicted = (y_val_pred_t4 == 2).sum()
delayed_tp        = ((y_val_pred_t4 == 2) & (y_test_arr_t4 == 2)).sum()
delayed_fn        = ((y_val_pred_t4 != 2) & (y_test_arr_t4 == 2)).sum()
delayed_fp        = ((y_val_pred_t4 == 2) & (y_test_arr_t4 != 2)).sum()

# Breakdown of how missed Delayed orders were misclassified
delayed_as_fast   = ((y_val_pred_t4 == 0) & (y_test_arr_t4 == 2)).sum()
delayed_as_normal = ((y_val_pred_t4 == 1) & (y_test_arr_t4 == 2)).sum()

print(f"   Actual Delayed orders          : {delayed_actual:>7,}")
print(f"   Predicted Delayed orders       : {delayed_predicted:>7,}")
print(f"   Correctly caught (TP)          : {delayed_tp:>7,}")
print(f"   Missed Delayed orders (FN)     : {delayed_fn:>7,}  "
      f"← no intervention triggered")
print(f"     └─ Misclassified as Fast     : {delayed_as_fast:>7,}  "
      f"← most severe miss")
print(f"     └─ Misclassified as Normal   : {delayed_as_normal:>7,}")
print(f"   False alarms (FP)              : {delayed_fp:>7,}")
if delayed_actual > 0:
    print(f"   Delayed Recall                 : {per_cls_rec_t4[2]:.4f}  "
          f"({'Good' if per_cls_rec_t4[2] > 0.5 else 'Needs improvement'})")
    print(f"   Delayed Precision              : {per_cls_pre_t4[2]:.4f}")

# ── V5. Normal class boundary analysis ───────────────────────────────────────
# Normal class (exactly 3 days) sits between Fast and Delayed
# Confusion at its boundaries reveals whether the model correctly identifies
print("\n   ── V5. Normal Class Boundary Analysis ───────────────────────────────")

normal_actual    = (y_test_arr_t4 == 1).sum()
normal_tp        = ((y_val_pred_t4 == 1) & (y_test_arr_t4 == 1)).sum()
normal_to_fast   = ((y_val_pred_t4 == 0) & (y_test_arr_t4 == 1)).sum()
normal_to_delayed= ((y_val_pred_t4 == 2) & (y_test_arr_t4 == 1)).sum()

print(f"   Actual Normal orders           : {normal_actual:>7,}")
print(f"   Correctly classified as Normal : {normal_tp:>7,}")
print(f"   Normal misclassified as Fast   : {normal_to_fast:>7,}  "
      f"(over-optimistic — understates processing time)")
print(f"   Normal misclassified as Delayed: {normal_to_delayed:>7,}  "
      f"(over-pessimistic — triggers unnecessary alerts)")
if normal_actual > 0:
    print(f"   Normal class accuracy          : "
          f"{100*normal_tp/normal_actual:.1f}%")

# ── V6. Prediction confidence per class ──────────────────────────────────────
print("\n   ── V6. Prediction Confidence per Class ──────────────────────────────")

for class_id, class_name in enumerate(CLASS_NAMES):
    class_probs  = y_val_prob_t4[:, class_id]
    pred_as_this = (y_val_pred_t4 == class_id)
    if pred_as_this.sum() > 0:
        mean_conf = class_probs[pred_as_this].mean()
        min_conf  = class_probs[pred_as_this].min()
        print(f"   {class_name:<20} — mean confidence: {mean_conf:.4f}  "
              f"min: {min_conf:.4f}")

# ── V7. Confidence distribution plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_colors = ["#43A047", "#FFA726", "#EF5350"]

for class_id, (class_name, color) in enumerate(
    zip(CLASS_NAMES, class_colors)
):
    ax = axes[class_id]
    ax.hist(
        y_val_prob_t4[:, class_id],
        bins=30, color=color,
        edgecolor="white", alpha=0.85
    )
    ax.axvline(0.5, color="black", linestyle="--", lw=1.2,
               label="0.5 threshold")
    ax.set_xlabel(f"P({class_name})")
    ax.set_ylabel("Frequency")
    ax.set_title(
        f"{class_name}\nF1={per_cls_f1_t4[class_id]:.4f}  "
        f"Recall={per_cls_rec_t4[class_id]:.4f}",
        fontsize=9, fontweight="bold"
    )
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(
    f"Task 4 — Prediction Confidence by Processing Class | {best_model_name}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "outputs_task4/task4_validation_confidence.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("\n   [OK] Saved outputs_task4/task4_validation_confidence.png")

# ── V8. Error pattern heatmap — predicted vs actual ───────────────────────────
# Shows exactly where the model confuses classes , more granular than
# the confusion matrix in Step 16 because it shows row-normalised rates
print("\n   ── V8. Error Pattern Analysis ───────────────────────────────────────")

cm_t4 = confusion_matrix(y_test_arr_t4, y_val_pred_t4)

# Row-normalised confusion matrix (recall per class)
cm_norm = cm_t4.astype(float) / cm_t4.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
disp_raw = ConfusionMatrixDisplay(
    confusion_matrix=cm_t4,
    display_labels=["Fast", "Normal", "Delayed"]
)
disp_raw.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(
    f"Confusion Matrix (Counts)\n{best_model_name}",
    fontsize=10, fontweight="bold"
)

# Row-normalised rates
sns.heatmap(
    cm_norm,
    annot=True, fmt=".3f",
    cmap="Blues",
    xticklabels=["Fast", "Normal", "Delayed"],
    yticklabels=["Fast", "Normal", "Delayed"],
    ax=axes[1],
    linewidths=0.5,
    annot_kws={"size": 10}
)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title(
    "Row-Normalised (Recall per Class)\nValues = fraction correctly classified",
    fontsize=10, fontweight="bold"
)

fig.suptitle(
    f"Task 4 — Error Pattern Analysis: {best_model_name}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "outputs_task4/task4_validation_error_pattern.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [OK] Saved outputs_task4/task4_validation_error_pattern.png")

# Print the normalised rates
print(f"\n   Row-normalised confusion rates:")
print(f"   {'Actual \\ Predicted':<22} {'→ Fast':>10} {'→ Normal':>10} {'→ Delayed':>10}")
print("   " + "-" * 55)
for i, name in enumerate(["Fast", "Normal", "Delayed"]):
    print(f"   {name:<22} {cm_norm[i,0]:>10.3f} {cm_norm[i,1]:>10.3f} {cm_norm[i,2]:>10.3f}")

# ── V9. Validation summary ────────────────────────────────────────────────────
print(f"""
   ── V9. Validation Summary ───────────────────────────────────────────────
   Best Model         : {best_model_name}
   Test Macro F1      : {val_mf1_t4:.4f}
   Test ROC-AUC       : {val_auc_t4:.4f}
   CV vs Test Gap     : {gap_t4:.4f}  ({verdict_t4.split(' — ')[0]})

   Per-Class Recall:
     Fast    (0-2d)   : {per_cls_rec_t4[0]:.4f}
     Normal  (3d)     : {per_cls_rec_t4[1]:.4f}
     Delayed (4-6d)   : {per_cls_rec_t4[2]:.4f}  ← most critical

   Delayed Class:
     Missed orders    : {delayed_fn:,}  (predicted Fast or Normal)
     Misclassified as Fast   : {delayed_as_fast:,}  (most severe miss)
     Misclassified as Normal : {delayed_as_normal:,}

   Normal Class Boundary:
     Confused as Fast    : {normal_to_fast:,}
     Confused as Delayed : {normal_to_delayed:,}

   ─────────────────────────────────────────────────────────────────────────
""")
print("=" * 90)

# ── 18. SHAP EXPLAINABILITY ────────────────────────────────────────────────────
# SHAP applied to the best model.
# For three-class problems, SHAP returns values per class — one summary plot
# per class helps to compare which features drive each processing tier.
# A global bar (mean |SHAP| across all classes) is also generated.

if SHAP_AVAILABLE:
    print("\n── 18. SHAP Explainability ──────────────────────────────────────────────")
    try:
        best_clf = best_pipeline.named_steps["clf"]
        prep     = best_pipeline.named_steps["prep"]

        N_SHAP        = min(500, X_test.shape[0])
        rng           = np.random.default_rng(RANDOM_STATE)
        idx           = rng.choice(X_test.shape[0], N_SHAP, replace=False)
        X_shap_sample = X_test.iloc[idx]

        X_shap_tr  = prep.transform(X_shap_sample)
        ohe_cols   = (prep.named_transformers_["cat"]["ohe"]
                      .get_feature_names_out(cat_feats).tolist())
        feat_names = num_feats + ohe_cols
        shap_df    = pd.DataFrame(X_shap_tr, columns=feat_names)

        if hasattr(best_clf, "feature_importances_"):
            explainer   = shap.TreeExplainer(best_clf)
            shap_values = explainer.shap_values(X_shap_tr)
        else:
            explainer   = shap.Explainer(best_clf, X_shap_tr)
            shap_values = explainer(X_shap_tr).values

        # Normalise to list of arrays — one per class
        if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            sv_list = [shap_values[:, :, c] for c in range(3)]
        elif isinstance(shap_values, list):
            sv_list = shap_values
        else:
            sv_list = [shap_values]

        # Per-class summary dot plots
        for cls_idx, cls_label in enumerate(CLASS_NAMES):
            if cls_idx >= len(sv_list):
                break
            sv = sv_list[cls_idx]

            plt.figure(figsize=(10, 7))
            shap.summary_plot(sv, shap_df, max_display=15,
                              show=False, plot_type="dot")
            plt.title(
                f"SHAP Summary — {best_model_name} | Class: {cls_label}",
                fontsize=11
            )
            plt.tight_layout()
            fname = f"outputs_task4/task4_shap_summary_class{cls_idx}.png"
            plt.savefig(fname, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"   [OK] Saved {fname}")

            # Bar plot per class
            plt.figure(figsize=(10, 6))
            shap.summary_plot(sv, shap_df, max_display=15,
                              show=False, plot_type="bar")
            plt.title(
                f"SHAP Feature Importance Bar — {cls_label}\n"
                f"(Model: {best_model_name})",
                fontsize=11
            )
            plt.tight_layout()
            fname_bar = f"outputs_task4/task4_shap_bar_class{cls_idx}.png"
            plt.savefig(fname_bar, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"   [OK] Saved {fname_bar}")

        # Global bar — mean |SHAP| across all three classes combined
        if len(sv_list) >= 3:
            mean_abs_shap     = np.mean([np.abs(sv_list[i]) for i in range(3)], axis=0)
            mean_per_feature  = mean_abs_shap.mean(axis=0)
            top15_idx         = np.argsort(mean_per_feature)[-15:][::-1]
            top15_feat        = [feat_names[i] for i in top15_idx]
            top15_vals        = mean_per_feature[top15_idx]

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.barh(top15_feat[::-1], top15_vals[::-1],
                    color="#1976D2", alpha=0.85)
            ax.set_xlabel("Mean |SHAP value| (averaged across all three classes)")
            ax.set_title(
                f"SHAP Global Feature Importance — {best_model_name} (Task 4)",
                fontsize=11, fontweight="bold"
            )
            ax.grid(axis="x", alpha=0.3)
            plt.tight_layout()
            plt.savefig("outputs_task4/task4_shap_global_bar.png",
                        dpi=150, bbox_inches="tight")
            plt.close()
            print("   [OK] Saved outputs_task4/task4_shap_global_bar.png")

    except Exception as e:
        print(f"   [!] SHAP failed for {best_model_name}: {e}")
        import traceback; traceback.print_exc()
else:
    print("\n── 18. SHAP — SKIPPED (shap not installed) ──────────────────────────────")

# ── 19. LIME EXPLAINABILITY ───────────────────────────────────────────────────
#   - One representative correctly predicted instance per class
#   - Probability scores printed in figure title
#   - Missed Delayed instance (highest operational cost error)
#   - CSV saved per class

print("\n── 19. LIME Explainability ──────────────────────────────────────────────")

X_train_encoded = X_train[all_features].copy()
X_test_encoded  = X_test[all_features].copy()

cat_indices       = [all_features.index(c) for c in cat_feats if c in all_features]
categorical_names = {}

for idx_c in cat_indices:
    col_name = all_features[idx_c]
    uniq     = list(X_train_encoded[col_name].astype(str).unique())
    categorical_names[idx_c] = uniq
    X_train_encoded[col_name] = X_train_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )
    X_test_encoded[col_name] = X_test_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )

X_train_np = X_train_encoded.values.astype(float)
X_test_np  = X_test_encoded.values.astype(float)

def lime_predict_fn(raw_rows: np.ndarray) -> np.ndarray:
    """Wraps the full pipeline for LIME; returns class probabilities (n x 3)."""
    df_temp = pd.DataFrame(raw_rows, columns=all_features)
    for idx_c in cat_indices:
        col_name = all_features[idx_c]
        uniq     = categorical_names[idx_c]
        df_temp[col_name] = df_temp[col_name].apply(
            lambda v: uniq[int(round(v))]
            if 0 <= int(round(v)) < len(uniq) else uniq[0]
        )
    if hasattr(best_pipeline, "predict_proba"):
        return best_pipeline.predict_proba(df_temp)
    else:
        scores = best_pipeline.decision_function(df_temp)
        exp_s  = np.exp(scores - scores.max(axis=1, keepdims=True))
        return exp_s / exp_s.sum(axis=1, keepdims=True)

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_np,
    feature_names         = all_features,
    class_names           = CLASS_NAMES,
    categorical_features  = cat_indices,
    categorical_names     = categorical_names,
    mode                  = "classification",
    random_state          = RANDOM_STATE,
    discretize_continuous = True,
)
print(f"   LIME explainer built on {X_train_np.shape[0]:,} training samples.")

y_pred_arr   = results[best_model_name]["_y_pred"]
y_prob_best  = results[best_model_name]["_y_prob"]
y_test_arr   = y_test.values

num_lime_features = min(15, len(all_features))
print(f"   LIME num_features set to: {num_lime_features}")

# ── 19a. One confidently correct instance per class ───────────────────────────
print("\n   Generating one local explanation per class (correct predictions)...")

for class_id, class_name in enumerate(CLASS_NAMES):
    correct_mask = (y_test_arr == class_id) & (y_pred_arr == class_id)
    correct_idxs = np.where(correct_mask)[0]

    if len(correct_idxs) == 0:
        any_idxs = np.where(y_test_arr == class_id)[0]
        if len(any_idxs) == 0:
            print(f"   [!] No instances found for Class {class_id} — skipping")
            continue
        chosen = any_idxs[0]
    else:
        # Most confident correct prediction
        chosen = correct_idxs[np.argmax(y_prob_best[correct_idxs, class_id])]

    instance_raw = X_test_np[chosen]
    prob_scores  = y_prob_best[chosen]

    exp = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = num_lime_features,
        labels       = (class_id,),
        num_samples  = 1000,
    )

    # Print top features
    print(f"\n   Top features — {class_name}:")
    for feat, weight in exp.as_list(label=class_id):
        print(f"   {feat:<45} {weight:>+.4f}")

    # Save feature weights to CSV
    lime_rows = []
    for feat, weight in exp.as_list(label=class_id):
        lime_rows.append({
            "Processing Class" : class_name,
            "Feature Condition": feat,
            "LIME Weight"      : round(weight, 4),
            "Direction"        : "increases delay" if weight > 0 else "decreases delay",
        })
    pd.DataFrame(lime_rows).to_csv(
        f"outputs_task4/task4_lime_{class_name.split('(')[0].strip().lower()}_features.csv",
        index=False
    )
    print(f"   [OK] Saved CSV for {class_name}")

    # Save HTML
    exp.save_to_file(
        f"outputs_task4/task4_lime_class{class_id}_correct.html"
    )

    # Save PNG — probability scores in title
    fig = exp.as_pyplot_figure(label=class_id)
    fig.set_size_inches(12, 6)
    fig.suptitle(
        f"LIME Local Explanation — {class_name} | Model: {best_model_name}\n"
        f"P(Fast)={prob_scores[0]:.3f}  "
        f"P(Normal)={prob_scores[1]:.3f}  "
        f"P(Delayed)={prob_scores[2]:.3f}",
        fontsize=10,
        fontweight="bold",
    )
    plt.tight_layout()
    png_path = f"outputs_task4/task4_lime_class{class_id}_correct.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   [OK] Saved {png_path}")

# ── 19b. Missed Delayed instance (highest operational cost error) ──────────────
# A Delayed order predicted as Fast or Normal is the most costly error:
print("\n   Generating explanation for missed Delayed instance...")

missed_delay_idx = np.where((y_pred_arr != 2) & (y_test_arr == 2))[0]
if len(missed_delay_idx) > 0:
    instance_raw = X_test_np[missed_delay_idx[0]]
    predicted_as = int(y_pred_arr[missed_delay_idx[0]])
    prob_scores  = y_prob_best[missed_delay_idx[0]]

    exp_fn = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = num_lime_features,
        num_samples  = 1000,
        labels       = (2,)     # explain why model missed Delayed
    )
    exp_fn.save_to_file("outputs_task4/task4_lime_missed_delayed.html")

    fig = exp_fn.as_pyplot_figure(label=2)
    fig.set_size_inches(12, 6)
    fig.suptitle(
        f"LIME — Missed Delayed Order (predicted as {CLASS_NAMES[predicted_as]})\n"
        f"Model: {best_model_name} | "
        f"P(Fast)={prob_scores[0]:.3f}  "
        f"P(Normal)={prob_scores[1]:.3f}  "
        f"P(Delayed)={prob_scores[2]:.3f}",
        fontsize=10,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("outputs_task4/task4_lime_missed_delayed.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("   [OK] Saved outputs_task4/task4_lime_missed_delayed.png")
else:
    print("   [!] No missed Delayed instances found in test set.")

print("\n   ── LIME Summary ──────────────────────────────────────────────────────")
print("   Local : One representative instance per class (Fast / Normal / Delayed)")
print("   Error : One missed Delayed instance (highest operational cost error)")
print("   LIME operates in raw feature space via pipeline wrapper.")

# ── 20. FINAL SUMMARY ────────────────────────────────────────────────────────
print("\n")
print("=" * 110)
print("  FINAL SUMMARY — TASK 4: ORDER PROCESSING DELAY CLASSIFICATION")
print("=" * 110)
print(comparison_df[[
    "Rank", "Model", "Accuracy", "Macro Precision", "Macro Recall",
    "Macro F1", "Weighted F1", "F1 Fast", "F1 Normal", "F1 Delayed",
    "ROC-AUC", "CV Macro F1 Mean", "CV Macro F1 Std", "Train Time (s)"
]].to_string(index=False))
print("=" * 110)
print(f"\n   Best Performing Model : {best_model_name}")
print(f"   Macro F1              : {best_macro_f1:.4f}")
print(f"   Accuracy              : {best_acc:.4f}")
print(f"   ROC-AUC               : {best_roc:.4f}")
print(f"\n   Saved outputs:")
print("    • outputs_task4/task4_target_distribution.png")
print("    • outputs_task4/task4_outlier_capping_log.csv")
print("    • outputs_task4/task4_model_comparison.csv")
print("    • outputs_task4/task4_model_comparison_bars.png")
print("    • outputs_task4/task4_per_class_f1.png")
print("    • outputs_task4/task4_heatmap.png")
print("    • outputs_task4/task4_confusion_matrix_best.png")
print("    • outputs_task4/task4_confusion_matrices_all.png")
print("    • outputs_task4/task4_roc_curves.png")
print("    • outputs_task4/task4_cv_macro_f1.png")
if SHAP_AVAILABLE:
    print("    • outputs_task4/task4_shap_summary_class[0-2].png")
    print("    • outputs_task4/task4_shap_bar_class[0-2].png")
    print("    • outputs_task4/task4_shap_global_bar.png")
print("    • outputs_task4/task4_lime_class[0-2]_correct.png/.html")
print("    • outputs_task4/task4_lime_missed_delayed.png/.html")
print("    • outputs_task4/task4_lime_*_features.csv")
print("=" * 110)


In [ ]:
from google.colab import files
!zip -r outputs_task4.zip /content/outputs_task4
files.download("outputs_task4.zip")